## Configuracion y Entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual para las gráficas
sns.set_theme(style="whitegrid", palette="Set2")

# 1. Cargar los datos
df = pd.read_csv('hotel-cancellation-prediction/data/raw/hotel_bookings.csv')

## Procesamiento de Datos

In [ ]:
#1. Manejo de Nulos
# Transformar la columna 'company' en binaria: 1 si hay compañía, 0 si no hay
df['is_corporate'] = df['company'].notna().astype(int)
df.drop('company', axis=1, inplace=True)

# Imputar nulos restantes con lógica de negocio
df['agent'] = df['agent'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['children'] = df['children'].fillna(0)

#2. Casting 

# Convertir variables flotantes anómalas a enteros
df['children'] = df['children'].astype('int64')
df['agent'] = df['agent'].astype('int64')

# Eliminar reservas fantasmas (reservas con 0 huespedes)
# Crear una condición donde haya al menos 1 huésped
filtro_huespedes = (df['adults'] > 0) | (df['children'] > 0) | (df['babies'] > 0)
# Aplicar el filtro para quedarnos solo con las reservas reales
df_clean = df[filtro_huespedes].copy()



## Explicacion y Descripcion de los datos


In [ ]:
# Creamos una figura con 2 subgráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafica 1: Distribución de la Variable Objetivo
sns.countplot(data=df_clean, x='is_canceled', hue='is_canceled', ax=axes[0], palette=['#2ecc71', '#e74c3c'], legend=False)
axes[0].set_title('A. Distribución de Cancelaciones (Target)', fontsize=14, fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Checkout Exitoso (0)', 'Cancelado (1)'])
axes[0].set_ylabel('Número de Reservas')
axes[0].set_xlabel('')

# Grafica 2: Lead Time vs Cancelaciones 
sns.boxplot(data=df_clean, x='is_canceled', y='lead_time', hue='is_canceled', ax=axes[1], palette=['#2ecc71', '#e74c3c'], legend=False)
axes[1].set_title('B. Días de Anticipación (Lead Time) vs Estado', fontsize=14, fontweight='bold')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Checkout Exitoso (0)', 'Cancelado (1)'])
axes[1].set_ylabel('Días de Anticipación (Lead Time)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

# Grafica 3: El impacto de la variable 'is_corporate' 
plt.figure(figsize=(9, 5))
sns.countplot(data=df_clean, x='is_corporate', hue='is_canceled', palette=['#2ecc71', '#e74c3c'])
plt.title('C. Comportamiento de Cancelación: Personal vs Corporativo', fontsize=14, fontweight='bold')
plt.xticks(ticks=[0, 1], labels=['Personal (0)', 'Corporativo (1)'])
plt.ylabel('Número de Reservas')
plt.xlabel('Tipo de Cliente')
plt.legend(title='Estado de Reserva', labels=['Checkout Exitoso', 'Cancelado'])
plt.show()

La primera grafica es la relacion de reservas canceladas y reservas exitosas, por la naturaleza del modelo que utilizaremos necesitamos comprobar si no hay un desbalance extremo para tomar medidas frente a esta situacion, en nuestro caso el desbalance no es tan grande como para tomar medidas drasticas como la creacion de datos sinteticos

La segunda es una grafica que relaciona las cancelaciones con el tiempo de reserva. Podemos comprobar que hay una relacion entre estas variables, ya que mietras menos tiempo pase (aproximadamente 40 dias) desde la llamada para reservar hasta la llegada menor es el riesgo, mientras que mientras mas tiempo pase (mas de 100 dias) los clientes suelen cambiar de opinion

La tercera es la relacion entre las reservas canceladas y reservas exitosas y la procedencia de la reserva (de una persona o de un corporativo). Esta es la variable is_corporate que creams y podemos notar que sera importante en un sector de la prediccion debido a su alta tasa de chekout exitoso en las reservas de corporativos. Entonces podemos concluir que aisalamos un comportamiento de consumo claro que nos puede ayudar en la prediccion futura.

